# Exercises 3 — Predictive Pipeline (Student)

In [ ]:
# ============================================================
# Part 0 — Environment & Dataset
# ============================================================
!pip -q install gradio gensim sentence-transformers transformers

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import math, random, warnings
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

BASE = Path("/content/drive/MyDrive/Colab Notebooks")
DATA_ROOT = BASE / "Dataset_Chapter3_4"

if (DATA_ROOT / "data").exists():
    DATA_DIR = DATA_ROOT / "data"
else:
    DATA_DIR = DATA_ROOT

print("BASE      =", BASE)
print("DATA_ROOT =", DATA_ROOT)
print("DATA_DIR  =", DATA_DIR)

items = pd.read_csv(DATA_DIR / "metadata.csv")
ratings = pd.read_csv(DATA_DIR / "ratings.csv")

rename_items = {
    "movie_id":"item_id", "id":"item_id",
    "movie_title":"title", "name":"title",
    "genres":"genre", "description":"synopsis",
    "plot":"synopsis", "summary":"synopsis"
}
for old,new in rename_items.items():
    if old in items.columns and new not in items.columns:
        items = items.rename(columns={old:new})

rename_r = {"uid":"user_id","user":"user_id","iid":"item_id","movie_id":"item_id","score":"rating"}
for old,new in rename_r.items():
    if old in ratings.columns and new not in ratings.columns:
        ratings = ratings.rename(columns={old:new})

required_i = ["item_id","title","genre","synopsis"]
required_r = ["user_id","item_id","rating"]
assert all(c in items.columns for c in required_i), items.columns.tolist()
assert all(c in ratings.columns for c in required_r), ratings.columns.tolist()

items["item_id"] = items["item_id"].astype(str)
ratings["item_id"] = ratings["item_id"].astype(str)
ratings["user_id"] = ratings["user_id"].astype(str)
ratings["rating"] = pd.to_numeric(ratings["rating"])

items["genre"] = items["genre"].fillna("").astype(str)
items["synopsis"] = items["synopsis"].fillna("").astype(str)
items["text_content"] = (items["genre"] + " " + items["synopsis"]).str.strip()

ITEM_INDEX = {iid:i for i,iid in enumerate(items["item_id"])}
N_USERS = max(1, ratings["user_id"].nunique())
POPULARITY = ratings.groupby("item_id")["user_id"].nunique().to_dict()

print(f"Items={len(items)}, Users={ratings['user_id'].nunique()}, Ratings={len(ratings)}")
display(items[["item_id","title","genre","synopsis"]].head())


# ============================================================
# Core Functions
# Vector Transformation + 3 User Profiles + Ranking + Metrics
# ============================================================

LIKED_THRESHOLD = 4.0
DISLIKED_THRESHOLD = 2.0

def transform_vectors(X, mode="Raw+L2"):
    X = np.asarray(X, dtype=float)
    if mode == "Raw+L2":
        return normalize(X)
    if mode == "Center+L2":
        return normalize(X - X.mean(axis=0, keepdims=True))
    if mode == "Standard+L2":
        return normalize(StandardScaler().fit_transform(X))
    raise ValueError(mode)

def _unit(v):
    v=np.asarray(v,dtype=float)
    n=np.linalg.norm(v)
    return v if n==0 else v/n

def profile_positive_only(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    idx=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    if not idx:
        idx=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
    return _unit(X[idx].mean(axis=0)) if idx else np.zeros(X.shape[1])

def profile_rating_centered(user_id, X, ratings_df):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)].copy()
    r=r[r["item_id"].astype(str).isin(ITEM_INDEX)]
    if r.empty: return np.zeros(X.shape[1])
    idx=np.array([ITEM_INDEX[str(i)] for i in r["item_id"]])
    w=r["rating"].astype(float).values - float(r["rating"].mean())
    denom=np.abs(w).sum()
    p=X[idx].mean(axis=0) if denom<=1e-12 else (X[idx]*w[:,None]).sum(axis=0)/denom
    return _unit(p)

def profile_rocchio(user_id, X, ratings_df, alpha=1.0, beta=1.0):
    r=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    liked=r[r["rating"]>=LIKED_THRESHOLD]
    disliked=r[r["rating"]<=DISLIKED_THRESHOLD]
    li=[ITEM_INDEX[str(i)] for i in liked["item_id"] if str(i) in ITEM_INDEX]
    di=[ITEM_INDEX[str(i)] for i in disliked["item_id"] if str(i) in ITEM_INDEX]
    pos=X[li].mean(axis=0) if li else np.zeros(X.shape[1])
    neg=X[di].mean(axis=0) if di else np.zeros(X.shape[1])
    p=alpha*pos-beta*neg
    if np.linalg.norm(p)==0:
        hi=[ITEM_INDEX[str(i)] for i in r["item_id"] if str(i) in ITEM_INDEX]
        p=X[hi].mean(axis=0) if hi else p
    return _unit(p)

PROFILE_METHODS = {
    "Positive-only": profile_positive_only,
    "Rating-centered": profile_rating_centered,
    "Rocchio": profile_rocchio,
}

def get_profile(user_id, X, ratings_df, method, alpha=1.0, beta=1.0):
    if method=="Rocchio":
        return profile_rocchio(user_id,X,ratings_df,alpha,beta)
    return PROFILE_METHODS[method](user_id,X,ratings_df)

def rank_user(user_id, X, ratings_df, method, k=None, candidates=None, alpha=1.0, beta=1.0):
    p=get_profile(user_id,X,ratings_df,method,alpha,beta)
    score=cosine_similarity(p.reshape(1,-1),X).ravel()
    seen=set(ratings_df.loc[ratings_df["user_id"].astype(str)==str(user_id),"item_id"].astype(str))
    if candidates is None:
        candidates=[iid for iid in items["item_id"] if iid not in seen]
    ranked=sorted(candidates,key=lambda iid:score[ITEM_INDEX[iid]],reverse=True)
    return ranked[:k] if k else ranked, score, p

# Accuracy / Ranking Metrics
def precision_at_k(recs, rel, k):
    return len(set(recs[:k]) & set(rel))/k if k else 0.0
def recall_at_k(recs, rel, k):
    rel=set(rel); return len(set(recs[:k])&rel)/len(rel) if rel else 0.0
def f1_at_k(recs, rel, k):
    p=precision_at_k(recs,rel,k); r=recall_at_k(recs,rel,k)
    return 0.0 if p+r==0 else 2*p*r/(p+r)
def hr_at_k(recs, rel, k):
    return float(bool(set(recs[:k])&set(rel)))
def reciprocal_rank(recs, rel):
    rel=set(rel)
    for rank,iid in enumerate(recs,1):
        if iid in rel: return 1.0/rank
    return 0.0
def ap_at_k(recs, rel, k):
    rel=set(rel); hits=0; s=0.0
    for i,iid in enumerate(recs[:k],1):
        if iid in rel:
            hits+=1; s+=hits/i
    return s/min(len(rel),k) if rel else 0.0
def ndcg_at_k(recs, rel, k):
    rel=set(rel)
    dcg=sum(1/math.log2(i+2) for i,iid in enumerate(recs[:k]) if iid in rel)
    idcg=sum(1/math.log2(i+2) for i in range(min(len(rel),k)))
    return dcg/idcg if idcg else 0.0

# Beyond-Accuracy Metrics
def coverage(all_recs):
    return len(set(all_recs))/len(items) if len(items) else 0.0

def gini_coefficient(all_recs):
    counts=pd.Series(all_recs).value_counts()
    x=np.array([counts.get(iid,0) for iid in items["item_id"]],dtype=float)
    if x.sum()==0: return 0.0
    x=np.sort(x); n=len(x)
    return float((2*np.sum(np.arange(1,n+1)*x)/(n*x.sum()))-(n+1)/n)

def ild(item_ids, X):
    idx=[ITEM_INDEX[i] for i in item_ids if i in ITEM_INDEX]
    if len(idx)<2: return 0.0
    D=1-cosine_similarity(X[idx])
    v=D[np.triu_indices_from(D,k=1)]
    return float(v.mean()) if len(v) else 0.0

def novelty(item_ids):
    vals=[]
    for iid in item_ids:
        pop=max(1,POPULARITY.get(iid,1))
        vals.append(-math.log2(pop/N_USERS))
    return float(np.mean(vals)) if vals else 0.0

def calibration_genre(user_id, rec_ids, ratings_df):
    hist=ratings_df[ratings_df["user_id"].astype(str)==str(user_id)]
    hist_genres=[]
    for iid in hist["item_id"].astype(str):
        if iid in ITEM_INDEX:
            hist_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    rec_genres=[]
    for iid in rec_ids:
        rec_genres += [g.strip() for g in str(items.iloc[ITEM_INDEX[iid]]["genre"]).replace("|",",").split(",") if g.strip()]
    if not hist_genres or not rec_genres: return np.nan
    cats=sorted(set(hist_genres)|set(rec_genres))
    hp=np.array([hist_genres.count(c) for c in cats],float); hp/=hp.sum()
    rp=np.array([rec_genres.count(c) for c in cats],float); rp/=rp.sum()
    return float(1-0.5*np.abs(hp-rp).sum())

def loo_folds(ratings_df, positive_threshold=4.0):
    folds=[]
    for uid,g in ratings_df.groupby("user_id"):
        pos=g[g["rating"]>=positive_threshold]
        if pos.empty: continue
        hold=pos.sort_values("rating").iloc[-1]
        folds.append((str(uid),ratings_df.drop(index=hold.name),str(hold["item_id"])))
    return folds

def evaluate_config(X, profile_method, k=3, alpha=1.0, beta=1.0, folds=None,
                    candidate_mode="Full Catalog", n_neg=20, seed=42):
    folds=folds or loo_folds(ratings)
    rng=np.random.default_rng(seed)
    rows=[]; global_recs=[]
    for uid,train,heldout in folds:
        seen=set(train.loc[train["user_id"].astype(str)==uid,"item_id"].astype(str))
        all_c=[iid for iid in items["item_id"] if iid not in seen]
        if heldout not in all_c: all_c.append(heldout)
        if candidate_mode=="Sampled Negatives":
            neg=[x for x in all_c if x!=heldout]
            sampled=list(rng.choice(neg,size=min(n_neg,len(neg)),replace=False)) if neg else []
            candidates=[heldout]+sampled
        else:
            candidates=all_c
        ranked,score,_=rank_user(uid,X,train,profile_method,candidates=candidates,alpha=alpha,beta=beta)
        topk=ranked[:k]; global_recs += topk
        y=np.array([1 if iid==heldout else 0 for iid in ranked])
        ys=np.array([score[ITEM_INDEX[iid]] for iid in ranked])
        auc=roc_auc_score(y,ys) if len(np.unique(y))==2 else np.nan
        rows.append({
            "user_id":uid,"heldout":heldout,
            f"Precision@{k}":precision_at_k(topk,[heldout],k),
            f"Recall@{k}":recall_at_k(topk,[heldout],k),
            f"F1@{k}":f1_at_k(topk,[heldout],k),
            f"HR@{k}":hr_at_k(topk,[heldout],k),
            "MRR":reciprocal_rank(ranked,[heldout]),
            f"MAP@{k}":ap_at_k(topk,[heldout],k),
            f"nDCG@{k}":ndcg_at_k(topk,[heldout],k),
            "AUC":auc,"ILD":ild(topk,X),"Novelty":novelty(topk),
            "Calibration":calibration_genre(uid,topk,train),
            "TopK":" | ".join(topk)
        })
    detail=pd.DataFrame(rows)
    summary=detail.select_dtypes(include=np.number).mean().to_dict() if len(detail) else {}
    summary["Coverage"]=coverage(global_recs)
    summary["Gini"]=gini_coefficient(global_recs)
    return pd.Series(summary), detail

## ข้อ 1 — เปลี่ยน User Profile
จาก pipeline ใน Lab 3 ให้เปลี่ยน `PROFILE_FOR_MODEL` จาก Rating-centered เป็น Positive-only และ Rocchio แล้ว rebuild pair features ก่อนวัด AUC

## ข้อ 2 — Hyperparameter
ทดลอง Logistic Regression ที่ `C = 0.1, 1, 10` โดยใช้ train/validation เดิม

## ข้อ 3 — Metric Thinking
ทำไม MAE/RMSE เหมาะกับ predicted rating แต่ไม่ควรใช้กับ classifier probability โดยตรง?

## ข้อ 4 — Data Leakage
ระบุจุดที่เกิด data leakage ได้อย่างน้อย 2 จุด และเสนอวิธีป้องกัน